In [13]:
#pandas for spreadsheet processing
import pandas as pd
pd.set_option('display.max_columns', None) #display all columns

#numpy for mathematical functions
import numpy as np

#geopandas to process geospatial/gis data
import geopandas as gpd

#library for connecting to the geodatabase
from sqlalchemy import create_engine, text
from geoalchemy2 import Geometry

# User input

- All human input values will need to be input in by the user into the following cells. 
- The rest of the code will run without required human intervention.

## 1. Input Link to downloaded datasets

In [14]:
# Religion
# To download original dataset go to - 
# https://www.ons.gov.uk/filters/a3287f4a-e115-428b-a2ce-2a72abe358a3/dimensions
religion_csv = r"C:\Users\abhimanya.achara\Downloads\TS030-2021-3.csv"

## 2. Input Link to LSOA 2021 shapefile
#### To download original dataset go to - 
https://geoportal.statistics.gov.uk/datasets/ons::lower-layer-super-output-areas-december-2021-boundaries-ew-bsc-v4-2/about

In [15]:
lsoa2021_shapefile = r"C:\Users\abhimanya.achara\Downloads\Local_Authority_Districts_December_2022_UK_BFE_V2_1999486602819488629\LAD_DEC_2022_UK_BFE_V2.shp"

## 3. Threshold to define dominant type
This defines the % threshold from the highest value of group of data columns to define which ones will be defined as dominant. 

In [16]:
threshold_value = 5

## 1. Import & Process Data

To download original dataset go to - 
https://www.nomisweb.co.uk/query/construct/summary.asp?mode=construct&version=0&dataset=2221

In [17]:
# Create Dataframe
lsoa_religion_df = pd.read_csv(religion_csv)
lsoa_religion_df.head()

,Lower tier local authorities Code,Lower tier local authorities,Religion (10 categories) Code,Religion (10 categories),Observation
0,E06000001,Hartlepool,-8,Does not apply,0
1,E06000001,Hartlepool,1,No religion,36995
2,E06000001,Hartlepool,2,Christian,48495
3,E06000001,Hartlepool,3,Buddhist,180
4,E06000001,Hartlepool,4,Hindu,222


In [18]:
#create pivot table
lsoa_religion_df_P = pd.pivot_table(lsoa_religion_df, values='Observation', index=['Lower tier local authorities Code'], columns=['Religion (10 categories)'], aggfunc=np.sum)
lsoa_religion_df_P = lsoa_religion_df_P.reset_index()

#drop columns
lsoa_religion_df_P.drop(['Does not apply'], 1, inplace=True)

# Rename columns: lowercase, replace spaces with _, remove colons, and add '_count' suffix
lsoa_religion_df_P.columns = (
    lsoa_religion_df_P.columns
    .str.lower()                 # Convert to lowercase
    .str.replace(' ', '_')       # Replace spaces with underscores
    .str.replace(':', '')        # Remove colons
    .str.cat(['_count'] * len(lsoa_religion_df_P.columns))  # Append '_count' to each column
)

#rename columns
lsoa_religion_df_P.rename(columns = {'lower_tier_local_authorities_code_count':'lad22cd'},inplace = True)
# Display the updated DataFrame

#create total population columns
lsoa_religion_df_P['total_religion_pop'] = lsoa_religion_df_P.sum(axis=1,numeric_only=True)

lsoa_religion_df_P.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_36816\2580649502.py:6: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  lsoa_religion_df_P.drop(['Does not apply'], 1, inplace=True)


Religion (10 categories),lad22cd,buddhist_count,christian_count,hindu_count,jewish_count,muslim_count,no_religion_count,not_answered_count,other_religion_count,sikh_count,total_religion_pop
0,E06000001,180,48495,222,27,1213,36995,4755,285,166,92338
1,E06000002,437,66143,1436,41,14703,52415,7683,460,606,143924
2,E06000003,280,72359,139,37,984,54921,7248,491,70,136529
3,E06000004,532,100420,811,61,6675,76840,9924,550,782,196595
4,E06000005,344,56194,453,36,1849,42780,5296,404,443,107799


## 2. Feature Engineering

In [19]:
# Create percentage columns for detailed ethnicity
total_religion_pop = 'total_religion_pop'
for col in lsoa_religion_df_P.columns[1:-1]:
    perc_col = col.replace('_count', '_perc')
    lsoa_religion_df_P[perc_col] = (lsoa_religion_df_P[col] / lsoa_religion_df_P[total_religion_pop]) * 100

lsoa_religion_df_P.head()

Religion (10 categories),lad22cd,buddhist_count,christian_count,hindu_count,jewish_count,muslim_count,no_religion_count,not_answered_count,other_religion_count,sikh_count,total_religion_pop,buddhist_perc,christian_perc,hindu_perc,jewish_perc,muslim_perc,no_religion_perc,not_answered_perc,other_religion_perc,sikh_perc
0,E06000001,180,48495,222,27,1213,36995,4755,285,166,92338,0.194936,52.519006,0.240421,0.029240,1.313652,40.064762,5.149559,0.308649,0.179774
1,E06000002,437,66143,1436,41,14703,52415,7683,460,606,143924,0.303632,45.956894,0.997749,0.028487,10.215808,36.418526,5.338234,0.319613,0.421056
2,E06000003,280,72359,139,37,984,54921,7248,491,70,136529,0.205085,52.998997,0.101810,0.027100,0.720726,40.226619,5.308762,0.359631,0.051271
3,E06000004,532,100420,811,61,6675,76840,9924,550,782,196595,0.270607,51.079631,0.412523,0.031028,3.395305,39.085429,5.047941,0.279763,0.397772
4,E06000005,344,56194,453,36,1849,42780,5296,404,443,107799,0.319112,52.128498,0.420227,0.033395,1.715229,39.684969,4.912847,0.374772,0.410950


## 4. Load GIS shapefile 

In [20]:
# Attempt to read from the server first, fallback to local path if it fails
lsoa2021boundary_df = gpd.read_file(lsoa2021_shapefile)
print("Shapefile loaded successfully from the server.")

# Display the first few rows
lsoa2021boundary_df.head()

Shapefile loaded successfully from the server.


,LAD22CD,LAD22NM,BNG_E,BNG_N,LONG,LAT,GlobalID,geometry
0,E06000001,Hartlepool,447160,531474,-1.27018,54.6761,ef33ae90-bd6b-43c8-aa11-b3308fb1b09d,"POLYGON ((447213.900 537036.104, 447228.798 53..."
1,E06000002,Middlesbrough,451141,516887,-1.21099,54.5447,408145f3-dcd6-455d-9dca-c1e11b5c9214,"POLYGON ((448489.897 522071.798, 448592.597 52..."
2,E06000003,Redcar and Cleveland,464361,519597,-1.00608,54.5675,4b8f6d25-4eb2-4ef4-8afb-5e20452921cd,"POLYGON ((455525.931 528406.654, 455724.632 52..."
3,E06000004,Stockton-on-Tees,444940,518183,-1.30664,54.5569,d978b075-04eb-4738-9611-fdc649760770,"POLYGON ((444157.002 527956.304, 444165.898 52..."
4,E06000005,Darlington,428029,515648,-1.56835,54.5353,eb7dbdf5-d3b5-4710-ab38-534d77fc4426,"POLYGON ((423496.602 524724.299, 423497.204 52..."


## 5. GIS data management

### Remove Rename fields

In [21]:
#Drop and rename fields
lsoa2021boundary_df.drop(['BNG_E', 'BNG_N', 'LONG','LAT','GlobalID',], 1, inplace = True)
lsoa2021boundary_df.rename(columns = {'LAD22CD':'lad22cd','LAD22NM':'lad22nm'}, inplace = True)
lsoa2021boundary_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_36816\111442805.py:2: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  lsoa2021boundary_df.drop(['BNG_E', 'BNG_N', 'LONG','LAT','GlobalID',], 1, inplace = True)


,lad22cd,lad22nm,geometry
0,E06000001,Hartlepool,"POLYGON ((447213.900 537036.104, 447228.798 53..."
1,E06000002,Middlesbrough,"POLYGON ((448489.897 522071.798, 448592.597 52..."
2,E06000003,Redcar and Cleveland,"POLYGON ((455525.931 528406.654, 455724.632 52..."
3,E06000004,Stockton-on-Tees,"POLYGON ((444157.002 527956.304, 444165.898 52..."
4,E06000005,Darlington,"POLYGON ((423496.602 524724.299, 423497.204 52..."


### Combine with boundary lookup table

#### LSOA21 to MSOA21 to LAD22

In [10]:
# Define the file path for msoa11 to msoa21 lookup
file_path = r"N:\Geodatabase\Raw_Data\Look up tables\OA21_LSOA21_MSOA21_LAD22_EW_LU.csv"

# Read the Excel file
lsoalookup_df = pd.read_csv(file_path)  # Reads the first sheet by default

#drop oa column
lsoalookup_df.drop(['oa21cd','lsoa21nm'],1,inplace = True)

#remove duplicates
lsoalookup_df = lsoalookup_df.drop_duplicates(subset='lsoa21cd', keep='first')

# Display the first few rows
lsoalookup_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_3860\1268765203.py:8: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  lsoalookup_df.drop(['oa21cd','lsoa21nm'],1,inplace = True)


,lsoa21cd,msoa21cd,msoa21nm,lad22cd,lad22nm
0,E01000001,E02000001,City of London 001,E09000001,City of London
4,E01000003,E02000001,City of London 001,E09000001,City of London
6,E01000002,E02000001,City of London 001,E09000001,City of London
12,E01032739,E02000001,City of London 001,E09000001,City of London
13,E01032740,E02000001,City of London 001,E09000001,City of London


In [11]:
lsoa2021boundary_df = pd.merge(lsoa2021boundary_df, lsoalookup_df, how = 'left', on = 'lsoa21cd')
lsoa2021boundary_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18...",E02000017,Barking and Dagenham 016,E09000002,Barking and Dagenham


#### LSOA21 to WARD22

In [12]:
# Define the file path for msoa21 to ward lookup
file_path = r"N:\Geodatabase\Raw_Data\Look up tables\Lower_Layer_Super_Output_Area_(2021)_to_Ward_(2022)_to_LAD_(2022)_Lookup_in_England_and_Wales_v3.csv"

# Read the Excel file
lsoawardlookup_df = pd.read_csv(file_path)  # Reads the first sheet by default

#drop unwanted fields
lsoawardlookup_df.drop(['LSOA21NM','LSOA21NMW','WD22NMW','LAD22NMW','ObjectId','LAD22CD','LAD22NM'],1,inplace = True)

# Rename fields
lsoawardlookup_df.rename(
    columns={
        'LSOA21CD': 'lsoa21cd',
        'WD22CD': 'wd22cd',
        'WD22NM': 'wd22nm',
        }, 
    inplace=True
)

# Display the first few rows
lsoawardlookup_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_3860\1707502962.py:8: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  lsoawardlookup_df.drop(['LSOA21NM','LSOA21NMW','WD22NMW','LAD22NMW','ObjectId','LAD22CD','LAD22NM'],1,inplace = True)


,lsoa21cd,wd22cd,wd22nm
0,E01004766,E05000650,Astley Bridge
1,E01005347,E05000722,Chadderton South
2,E01004890,E05000661,Horwich North East
3,E01004770,E05000650,Astley Bridge
4,E01004888,E05000661,Horwich North East


In [13]:
lsoa2021boundary_df = pd.merge(lsoa2021boundary_df, lsoawardlookup_df, how = 'inner', on = 'lsoa21cd')
lsoa2021boundary_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London,E05009288,Aldersgate
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London,E05009308,Portsoken
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18...",E02000017,Barking and Dagenham 016,E09000002,Barking and Dagenham,E05014066,Northbury


#### LAD22 to REGION22

In [22]:
# Define the file path for msoa21 to ward lookup
file_path = r"N:\Geodatabase\Raw_Data\Look up tables\Local_Authority_District_to_Region_(December_2022)_Lookup_in_England.csv"

# Read the Excel file
ladregionlookup_df = pd.read_csv(file_path)  # Reads the first sheet by default

#drop unwanted fields
ladregionlookup_df.drop(['LAD22NM','ObjectId'],1,inplace = True)

# Rename fields
ladregionlookup_df.rename(
    columns={
        'LAD22CD': 'lad22cd',
        'RGN22CD': 'rgn22cd',
        'RGN22NM': 'rgn22nm'
       }, 
    inplace=True
)

# Display the first few rows
ladregionlookup_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_36816\4188529333.py:8: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  ladregionlookup_df.drop(['LAD22NM','ObjectId'],1,inplace = True)


,lad22cd,rgn22cd,rgn22nm
0,E06000001,E12000001,North East
1,E06000002,E12000001,North East
2,E06000003,E12000001,North East
3,E06000004,E12000001,North East
4,E06000005,E12000001,North East


In [23]:
lsoa2021boundary_df = pd.merge(lsoa2021boundary_df, ladregionlookup_df, how = 'left', on = 'lad22cd')
lsoa2021boundary_df.head()

,lad22cd,lad22nm,geometry,rgn22cd,rgn22nm
0,E06000001,Hartlepool,"POLYGON ((447213.900 537036.104, 447228.798 53...",E12000001,North East
1,E06000002,Middlesbrough,"POLYGON ((448489.897 522071.798, 448592.597 52...",E12000001,North East
2,E06000003,Redcar and Cleveland,"POLYGON ((455525.931 528406.654, 455724.632 52...",E12000001,North East
3,E06000004,Stockton-on-Tees,"POLYGON ((444157.002 527956.304, 444165.898 52...",E12000001,North East
4,E06000005,Darlington,"POLYGON ((423496.602 524724.299, 423497.204 52...",E12000001,North East


### Add data management fields

In [16]:
# Add new data management fields with specified values
lsoa2021boundary_df['data_source'] = "Census 2021 (ONS Nomis - Official Census and Labour Market Statistics)"
lsoa2021boundary_df['data_resolution'] = "LSOA 2021"
lsoa2021boundary_df['data_time_period'] = pd.to_datetime("2021-02-21")  # Convert to date format
lsoa2021boundary_df['data_web_link'] = "https://www.nomisweb.co.uk/"
lsoa2021boundary_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London,E05009288,Aldersgate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London,E05009308,Portsoken,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18...",E02000017,Barking and Dagenham 016,E09000002,Barking and Dagenham,E05014066,Northbury,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/


### Update Area

In [24]:
#Update Area in Hectares
# Ensure the dataframe has a valid geometry column
if 'geometry' in lsoa2021boundary_df.columns:
    # Calculate the area in square meters and convert to hectares (1 hectare = 10,000 m²)
    lsoa2021boundary_df['area_ha'] = lsoa2021boundary_df.geometry.area / 10_000

    # Print confirmation message
    print("Field 'area_ha' has been added and updated successfully.")
else:
    print("Error: No 'geometry' column found in lsoa2021boundary_df. Ensure it's a GeoDataFrame with valid geometries.")
    
lsoa2021boundary_df.head()

Field 'area_ha' has been added and updated successfully.


,lad22cd,lad22nm,geometry,rgn22cd,rgn22nm,area_ha
0,E06000001,Hartlepool,"POLYGON ((447213.900 537036.104, 447228.798 53...",E12000001,North East,9835.106455
1,E06000002,Middlesbrough,"POLYGON ((448489.897 522071.798, 448592.597 52...",E12000001,North East,5455.358107
2,E06000003,Redcar and Cleveland,"POLYGON ((455525.931 528406.654, 455724.632 52...",E12000001,North East,25378.535949
3,E06000004,Stockton-on-Tees,"POLYGON ((444157.002 527956.304, 444165.898 52...",E12000001,North East,20973.080153
4,E06000005,Darlington,"POLYGON ((423496.602 524724.299, 423497.204 52...",E12000001,North East,19747.775356


# 7. Combine Geometry and data

In [40]:
census2021_religion_lsoa2021_gdb_df = pd.merge(lsoa2021boundary_df,lsoa_religion_df_P, how = 'left', on='lad22cd')
census2021_religion_lsoa2021_gdb_df.head()

,lad22cd,lad22nm,geometry,rgn22cd,rgn22nm,area_ha,buddhist_count,christian_count,hindu_count,jewish_count,muslim_count,no_religion_count,not_answered_count,other_religion_count,sikh_count,total_religion_pop,buddhist_perc,christian_perc,hindu_perc,jewish_perc,muslim_perc,no_religion_perc,not_answered_perc,other_religion_perc,sikh_perc
0,E06000001,Hartlepool,"POLYGON ((447213.900 537036.104, 447228.798 53...",E12000001,North East,9835.106455,180.0,48495.0,222.0,27.0,1213.0,36995.0,4755.0,285.0,166.0,92338.0,0.194936,52.519006,0.240421,0.029240,1.313652,40.064762,5.149559,0.308649,0.179774
1,E06000002,Middlesbrough,"POLYGON ((448489.897 522071.798, 448592.597 52...",E12000001,North East,5455.358107,437.0,66143.0,1436.0,41.0,14703.0,52415.0,7683.0,460.0,606.0,143924.0,0.303632,45.956894,0.997749,0.028487,10.215808,36.418526,5.338234,0.319613,0.421056
2,E06000003,Redcar and Cleveland,"POLYGON ((455525.931 528406.654, 455724.632 52...",E12000001,North East,25378.535949,280.0,72359.0,139.0,37.0,984.0,54921.0,7248.0,491.0,70.0,136529.0,0.205085,52.998997,0.101810,0.027100,0.720726,40.226619,5.308762,0.359631,0.051271
3,E06000004,Stockton-on-Tees,"POLYGON ((444157.002 527956.304, 444165.898 52...",E12000001,North East,20973.080153,532.0,100420.0,811.0,61.0,6675.0,76840.0,9924.0,550.0,782.0,196595.0,0.270607,51.079631,0.412523,0.031028,3.395305,39.085429,5.047941,0.279763,0.397772
4,E06000005,Darlington,"POLYGON ((423496.602 524724.299, 423497.204 52...",E12000001,North East,19747.775356,344.0,56194.0,453.0,36.0,1849.0,42780.0,5296.0,404.0,443.0,107799.0,0.319112,52.128498,0.420227,0.033395,1.715229,39.684969,4.912847,0.374772,0.410950


In [41]:
# Create a FID starting from 1 to the length of the dataframe
census2021_religion_lsoa2021_gdb_df['FID'] = range(1, len(census2021_religion_lsoa2021_gdb_df) + 1)
census2021_religion_lsoa2021_gdb_df.head()

,lad22cd,lad22nm,geometry,rgn22cd,rgn22nm,area_ha,buddhist_count,christian_count,hindu_count,jewish_count,muslim_count,no_religion_count,not_answered_count,other_religion_count,sikh_count,total_religion_pop,buddhist_perc,christian_perc,hindu_perc,jewish_perc,muslim_perc,no_religion_perc,not_answered_perc,other_religion_perc,sikh_perc,FID
0,E06000001,Hartlepool,"POLYGON ((447213.900 537036.104, 447228.798 53...",E12000001,North East,9835.106455,180.0,48495.0,222.0,27.0,1213.0,36995.0,4755.0,285.0,166.0,92338.0,0.194936,52.519006,0.240421,0.029240,1.313652,40.064762,5.149559,0.308649,0.179774,1
1,E06000002,Middlesbrough,"POLYGON ((448489.897 522071.798, 448592.597 52...",E12000001,North East,5455.358107,437.0,66143.0,1436.0,41.0,14703.0,52415.0,7683.0,460.0,606.0,143924.0,0.303632,45.956894,0.997749,0.028487,10.215808,36.418526,5.338234,0.319613,0.421056,2
2,E06000003,Redcar and Cleveland,"POLYGON ((455525.931 528406.654, 455724.632 52...",E12000001,North East,25378.535949,280.0,72359.0,139.0,37.0,984.0,54921.0,7248.0,491.0,70.0,136529.0,0.205085,52.998997,0.101810,0.027100,0.720726,40.226619,5.308762,0.359631,0.051271,3
3,E06000004,Stockton-on-Tees,"POLYGON ((444157.002 527956.304, 444165.898 52...",E12000001,North East,20973.080153,532.0,100420.0,811.0,61.0,6675.0,76840.0,9924.0,550.0,782.0,196595.0,0.270607,51.079631,0.412523,0.031028,3.395305,39.085429,5.047941,0.279763,0.397772,4
4,E06000005,Darlington,"POLYGON ((423496.602 524724.299, 423497.204 52...",E12000001,North East,19747.775356,344.0,56194.0,453.0,36.0,1849.0,42780.0,5296.0,404.0,443.0,107799.0,0.319112,52.128498,0.420227,0.033395,1.715229,39.684969,4.912847,0.374772,0.410950,5


# 8. Final tweaks

In [42]:
# Reorder columns in the combined dataframe

final_column_order = ['FID', 
 'lad22cd',
 'lad22nm',
'geometry',
 'rgn22cd',
 'rgn22nm', 
 'area_ha',
 'buddhist_count',
 'christian_count',
 'hindu_count',
 'jewish_count',
 'muslim_count',
 'no_religion_count',
 'not_answered_count',
 'other_religion_count',
 'sikh_count',
 'total_religion_pop',
 'buddhist_perc',
 'christian_perc',
 'hindu_perc',
 'jewish_perc',
 'muslim_perc',
 'no_religion_perc',
 'not_answered_perc',
 'other_religion_perc',
 'sikh_perc']

census2021_religion_lsoa2021_gdb_df = census2021_religion_lsoa2021_gdb_df[final_column_order]

census2021_religion_lsoa2021_gdb_df.head()

,FID,lad22cd,lad22nm,geometry,rgn22cd,rgn22nm,area_ha,buddhist_count,christian_count,hindu_count,jewish_count,muslim_count,no_religion_count,not_answered_count,other_religion_count,sikh_count,total_religion_pop,buddhist_perc,christian_perc,hindu_perc,jewish_perc,muslim_perc,no_religion_perc,not_answered_perc,other_religion_perc,sikh_perc
0,1,E06000001,Hartlepool,"POLYGON ((447213.900 537036.104, 447228.798 53...",E12000001,North East,9835.106455,180.0,48495.0,222.0,27.0,1213.0,36995.0,4755.0,285.0,166.0,92338.0,0.194936,52.519006,0.240421,0.029240,1.313652,40.064762,5.149559,0.308649,0.179774
1,2,E06000002,Middlesbrough,"POLYGON ((448489.897 522071.798, 448592.597 52...",E12000001,North East,5455.358107,437.0,66143.0,1436.0,41.0,14703.0,52415.0,7683.0,460.0,606.0,143924.0,0.303632,45.956894,0.997749,0.028487,10.215808,36.418526,5.338234,0.319613,0.421056
2,3,E06000003,Redcar and Cleveland,"POLYGON ((455525.931 528406.654, 455724.632 52...",E12000001,North East,25378.535949,280.0,72359.0,139.0,37.0,984.0,54921.0,7248.0,491.0,70.0,136529.0,0.205085,52.998997,0.101810,0.027100,0.720726,40.226619,5.308762,0.359631,0.051271
3,4,E06000004,Stockton-on-Tees,"POLYGON ((444157.002 527956.304, 444165.898 52...",E12000001,North East,20973.080153,532.0,100420.0,811.0,61.0,6675.0,76840.0,9924.0,550.0,782.0,196595.0,0.270607,51.079631,0.412523,0.031028,3.395305,39.085429,5.047941,0.279763,0.397772
4,5,E06000005,Darlington,"POLYGON ((423496.602 524724.299, 423497.204 52...",E12000001,North East,19747.775356,344.0,56194.0,453.0,36.0,1849.0,42780.0,5296.0,404.0,443.0,107799.0,0.319112,52.128498,0.420227,0.033395,1.715229,39.684969,4.912847,0.374772,0.410950


# 8. Create dominant field

In [43]:
def determine_dominant_group(row):
    age_columns = [
 'buddhist_perc',
 'christian_perc',
 'hindu_perc',
 'jewish_perc',
 'muslim_perc',
 'no_religion_perc',
 'not_answered_perc',
 'other_religion_perc',
 'sikh_perc'
    ]
    
    max_value = max(row[col] for col in age_columns)
    threshold = max_value - (threshold_value)
    
    dominant_groups = [col.replace('_perc', '') for col in age_columns if row[col] >= threshold]
    
    return ', '.join(dominant_groups)

census2021_religion_lsoa2021_gdb_df['dominant_relegious_group'] = census2021_religion_lsoa2021_gdb_df.apply(determine_dominant_group, axis=1)

# Display the first few rows of the updated dataframe
census2021_religion_lsoa2021_gdb_df.head()

,FID,lad22cd,lad22nm,geometry,rgn22cd,rgn22nm,area_ha,buddhist_count,christian_count,hindu_count,jewish_count,muslim_count,no_religion_count,not_answered_count,other_religion_count,sikh_count,total_religion_pop,buddhist_perc,christian_perc,hindu_perc,jewish_perc,muslim_perc,no_religion_perc,not_answered_perc,other_religion_perc,sikh_perc,dominant_relegious_group
0,1,E06000001,Hartlepool,"POLYGON ((447213.900 537036.104, 447228.798 53...",E12000001,North East,9835.106455,180.0,48495.0,222.0,27.0,1213.0,36995.0,4755.0,285.0,166.0,92338.0,0.194936,52.519006,0.240421,0.029240,1.313652,40.064762,5.149559,0.308649,0.179774,christian
1,2,E06000002,Middlesbrough,"POLYGON ((448489.897 522071.798, 448592.597 52...",E12000001,North East,5455.358107,437.0,66143.0,1436.0,41.0,14703.0,52415.0,7683.0,460.0,606.0,143924.0,0.303632,45.956894,0.997749,0.028487,10.215808,36.418526,5.338234,0.319613,0.421056,christian
2,3,E06000003,Redcar and Cleveland,"POLYGON ((455525.931 528406.654, 455724.632 52...",E12000001,North East,25378.535949,280.0,72359.0,139.0,37.0,984.0,54921.0,7248.0,491.0,70.0,136529.0,0.205085,52.998997,0.101810,0.027100,0.720726,40.226619,5.308762,0.359631,0.051271,christian
3,4,E06000004,Stockton-on-Tees,"POLYGON ((444157.002 527956.304, 444165.898 52...",E12000001,North East,20973.080153,532.0,100420.0,811.0,61.0,6675.0,76840.0,9924.0,550.0,782.0,196595.0,0.270607,51.079631,0.412523,0.031028,3.395305,39.085429,5.047941,0.279763,0.397772,christian
4,5,E06000005,Darlington,"POLYGON ((423496.602 524724.299, 423497.204 52...",E12000001,North East,19747.775356,344.0,56194.0,453.0,36.0,1849.0,42780.0,5296.0,404.0,443.0,107799.0,0.319112,52.128498,0.420227,0.033395,1.715229,39.684969,4.912847,0.374772,0.410950,christian


# 9. Upload to geodatabase

In [44]:
# Define database connection parameters
db_host = "PRIORPSRV03"
db_name = "gis"
db_port = "5432"
db_schema = "pp_sds"
table_name = "census2021_lad2022_religion"  # Desired table name
primary_key_column = "lad22cd"  # Define the primary key column (change based on your dataset)
geometry_column = "geometry"  # Default geometry column
# Create the database connection string (Windows Authentication - Trusted Connection)
conn_str = f"postgresql+psycopg2://@{db_host}:{db_port}/{db_name}?sslmode=disable"

# Create a SQLAlchemy engine
engine = create_engine(conn_str)

In [45]:
# Ensure the GeoDataFrame has a valid CRS before writing
if census2021_religion_lsoa2021_gdb_df.crs is None:
    print("Warning: GeoDataFrame has no CRS. Setting default to EPSG:27700 (British National Grid).")
    census2021_religion_lsoa2021_gdb_df.set_crs(epsg=27700, inplace=True)

In [46]:
# Publish the GeoDataFrame to PostGIS
census2021_religion_lsoa2021_gdb_df.to_postgis(
    name=table_name,
    con=engine,
    schema=db_schema,
    if_exists="replace",
    index=False,
    dtype={'geometry': 'MULTIPOLYGON'}  # Ensure geometry is stored as MULTIPOLYGON
)

print(f"Data successfully uploaded to PostGIS: {db_schema}.{table_name}")

# Connect to the database to modify table structure
with engine.connect() as conn:
    # Set Primary Key (if it doesn't exist already)
    alter_pk_query = text(f"""
        ALTER TABLE {db_schema}.{table_name}
        ADD CONSTRAINT {table_name}_pkey PRIMARY KEY ({primary_key_column});
    """)
    
    # Create Spatial Index
    create_spatial_index_query = text(f"""
        CREATE INDEX {table_name}_geom_idx
        ON {db_schema}.{table_name}
        USING GIST ({geometry_column});
    """)

    try:
        conn.execute(alter_pk_query)  # Add Primary Key
        print(f"Primary key set on column: {primary_key_column}")
    except Exception as e:
        print(f"Could not set primary key. It may already exist. Error: {e}")

    try:
        conn.execute(create_spatial_index_query)  # Add Spatial Index
        print(f"Spatial index created for geometry column: {geometry_column}")
    except Exception as e:
        print(f"Could not create spatial index. It may already exist. Error: {e}")

print(f"GeoDataFrame successfully published to PostGIS with Primary Key and Spatial Index: {db_schema}.{table_name}")

Data successfully uploaded to PostGIS: pp_sds.census2021_lad2022_religion
Primary key set on column: lad22cd
Spatial index created for geometry column: geometry
GeoDataFrame successfully published to PostGIS with Primary Key and Spatial Index: pp_sds.census2021_lad2022_religion
